# 06 · Transformers and Attention Basics

In plain English, this notebook is about the one idea that powers every modern large language model: **attention**. Attention is the trick that lets a model *look at the other words in a sentence to figure out what each word actually means*. The word "bank" means something very different next to "river" than next to "money" — and attention is how the model notices. Once you understand attention, the rest of a Transformer (the architecture behind GPT, BERT, Llama, and friends) is just a tidy way of stacking that idea many times. We'll build a tiny attention calculation by hand in PyTorch so it stops being magic and starts being arithmetic.

## What you'll learn

- **The problem attention solves:** why a word's meaning depends on its neighbors, and why older models struggled with that.
- **Self-attention intuition:** the **Query / Key / Value** idea explained as a *soft dictionary lookup* — every token asks a question and listens to the answers of every other token.
- **Scaled dot-product attention, step by step** in PyTorch on a tiny example (4 tokens, embedding dim 8): scores → scale → `softmax` → weighted values.
- **Softmax**, explained simply: how to turn raw match scores into weights that sum to 1.
- **Multi-head attention:** running several attentions in parallel so the model can track several relationships at once.
- **The Transformer block:** attention + feed-forward + residual connections + layer norm, and why stacking blocks makes a deep model.
- **Encoder vs decoder:** BERT-style models (great for understanding/classification) vs GPT-style models (great for generation), plus **causal masking**.
- **How this connects to fine-tuning:** what weights actually change when you fine-tune.

## Why this matters for fine-tuning

A large language model is, almost literally, a **tall stack of Transformer blocks**. Nothing more exotic than that, repeated dozens of times. When you *fine-tune* a model, you are nudging the numbers (the **weights**) that live inside those blocks — mostly inside the **attention** layers and the **feed-forward** layers we'll meet here.

That has three practical consequences for the rest of this course:

- **Full fine-tuning** adjusts *all* of those weights — powerful, but memory-hungry.
- **LoRA and other parameter-efficient methods** (coming later) leave the original weights frozen and bolt **small trainable pieces** onto the attention and feed-forward layers instead. To understand *where* LoRA attaches, you first need to know what attention and feed-forward layers *are* — that's this notebook.
- When something goes wrong in training, knowing the architecture helps you reason about it ("the model isn't attending to the right tokens", "this is a decoder, so it can't see the future").

You do **not** need to be able to build a Transformer from scratch. You need a clear mental picture. That's the goal here.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment that doesn't already have these libraries. Everything here runs comfortably on a **CPU**; we use tiny examples on purpose.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install torch numpy matplotlib

import math                       # for sqrt() when we scale attention scores
import torch                      # PyTorch: tensors + neural network building blocks
import torch.nn.functional as F   # F.softmax and friends live here
import numpy as np                # handy for a couple of small array ops
import matplotlib.pyplot as plt   # to visualize the attention weight matrix

torch.manual_seed(0)              # make random numbers reproducible across runs
print("Imports OK")               # -> Imports OK

## 1. The problem attention solves: meaning depends on context

Read these two sentences:

1. "I sat by the river **bank**."
2. "I deposited cash at the **bank**."

The word **bank** is spelled identically both times, but it means two different things. The *only* way to tell them apart is to look at the **other words** in the sentence ("river" vs "cash", "deposited"). A model that processes each word in isolation has no hope.

Older approaches (like reading a sentence strictly left-to-right, one word at a time) struggled here: by the time the model reached "bank", information about "river" had to survive a long, lossy journey, and the model couldn't easily jump *back* to look at it.

**Attention fixes this directly.** It lets every token *look at every other token* and pull in the information it needs — no matter how far away. "bank" can reach over and read "river" instantly. Let's build the intuition before the math.

In [ ]:
# A toy "sentence" of 4 tokens. We'll reuse this everywhere below.
tokens = ["the", "river", "bank", "flooded"]
print("Our tiny sentence:", tokens)
print("Number of tokens:", len(tokens))

# Intuition: when the model processes "bank", we WANT it to look mostly at
# "river" (which disambiguates the meaning) and at "flooded" (the action).
# Attention is the mechanism that produces exactly that "look mostly at..." behavior.
# -> Our tiny sentence: ['the', 'river', 'bank', 'flooded']
# -> Number of tokens: 4

**What this does:** It sets up a 4-word toy sentence we'll carry through the whole notebook. There's no model yet — we're just naming the goal: when the model handles `"bank"`, attention should make it *focus on the helpful neighbors* (`"river"`, `"flooded"`) rather than the unhelpful ones (`"the"`).

### ✏️ Exercise
Think of your own ambiguous word (e.g. "bat", "spring", "match"). Write two short sentences where it means different things, and note which neighboring word resolves the ambiguity. You can edit `tokens` above to a different 4-word sentence and re-run — everything downstream will still work.

## 2. First, tokens become vectors (embeddings)

A model can't do math on the *string* `"bank"`. So step one (covered in depth in the next notebook) is turning each token into a list of numbers called an **embedding vector**. Think of it as the token's coordinates in "meaning space".

For this notebook we'll just make up small random embeddings — **4 tokens, each an 8-dimensional vector** — so we can focus on attention itself. Real models learn these numbers during training; ours are random placeholders, which is fine for seeing the *mechanics*.

In [ ]:
n_tokens = 4   # our sentence has 4 tokens
d_model  = 8   # each token is represented by an 8-number vector (kept tiny on purpose)

# X holds the input embeddings: one row per token, one column per feature.
# Shape (4, 8) reads as "4 tokens, each with 8 numbers".
X = torch.randn(n_tokens, d_model)

print("X shape:", X.shape)          # -> X shape: torch.Size([4, 8])
print("Embedding for 'bank' (row 2):")
print(X[2])                          # the 8 numbers standing in for "bank"

**What this does:** It creates a `(4, 8)` tensor `X`. Each **row** is one token's embedding; each row has 8 numbers. `X[2]` is the embedding for `"bank"` (rows are 0-indexed: `the`=0, `river`=1, `bank`=2, `flooded`=3). These numbers are random here, but the *shapes* are exactly what a real model uses — just much smaller.

### ✏️ Exercise
Print `X.shape` and `X[1]` (the `"river"` row). Then change `d_model` to `4` and re-run this cell. Confirm the shape becomes `(4, 4)`. Smaller dimensions are easier to read while you learn.

## 3. Self-attention as a soft dictionary lookup: Query, Key, Value

Here's the core mental model. Imagine a **search engine** running *inside* the sentence. For every token we compute three different vectors:

- **Query (Q)** — the *question* this token is asking. ("I'm 'bank' — which other words help define me?")
- **Key (K)** — a *label* each token advertises about itself, so others can decide whether it's relevant. ("I'm 'river', here's what I'm about.")
- **Value (V)** — the *actual information* a token will hand over if it's chosen. ("If you attend to me, here's the content you get.")

The process is a **soft dictionary lookup**:

1. Each token's **Query** is compared against **every** token's **Key** to get a *match score*.
2. High score = "very relevant", low score = "ignore". We turn those scores into **weights that sum to 1** (with softmax).
3. The token's new representation is a **blend of all the Values**, weighted by those scores.

"Soft" means we don't pick a single best match — we mix in a *little of everything*, mostly the relevant stuff. We get Q, K, and V by multiplying the embeddings by three learned weight matrices.

In [ ]:
# Three weight matrices turn each 8-dim embedding into an 8-dim Q, K, and V.
# In a real model these are LEARNED during training; here they're random.
# Shape (d_model, d_model) = (8, 8): they map an 8-vector to an 8-vector.
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

# Apply them to every token at once via matrix multiplication (the @ operator).
# X is (4, 8); each W is (8, 8); so Q, K, V come out (4, 8).
Q = X @ W_q   # one query vector per token
K = X @ W_k   # one key   vector per token
V = X @ W_v   # one value vector per token

print("Q shape:", Q.shape)   # -> Q shape: torch.Size([4, 8])
print("K shape:", K.shape)   # -> K shape: torch.Size([4, 8])
print("V shape:", V.shape)   # -> V shape: torch.Size([4, 8])

**What this does:** It builds `Q`, `K`, `V` — one of each *per token* — by multiplying the inputs `X` by three weight matrices. The `@` symbol is matrix multiplication. Because `X` is `(4, 8)` and each weight matrix is `(8, 8)`, each result is `(4, 8)`: 4 tokens, each now described by an 8-number query, key, and value. **In a real Transformer these three weight matrices are exactly the kind of thing fine-tuning adjusts.**

### ✏️ Exercise
What is the shape of `W_q`? What would happen to the shape of `Q` if `W_q` were `(8, 4)` instead of `(8, 8)`? (Hint: matrix multiply `(4, 8) @ (8, 4)` gives `(4, 4)`.) Try it by editing one matrix and re-running, then change it back.

## 4. Step 1 — match scores: compare every Query with every Key

Now we ask: *how well does each token's Query match each token's Key?* The natural way to compare two vectors is the **dot product**: multiply them element-by-element and add it all up. A **big** dot product means the two vectors point in a similar direction → "these two are relevant to each other." A small (or negative) dot product means "not so relevant."

We want this comparison for **every (query, key) pair** — all 4 × 4 = 16 of them. The compact way to compute all pairs at once is `Q @ K.T` (Q times the *transpose* of K). The result is a **4 × 4 grid of scores**, where row *i*, column *j* answers: *"how much should token i pay attention to token j?"*

In [ ]:
# K.T is the "transpose": it flips K from (4, 8) to (8, 4) so the shapes line up
# for matrix multiplication. (4, 8) @ (8, 4) -> (4, 4): every query vs every key.
scores = Q @ K.T

print("scores shape:", scores.shape)   # -> scores shape: torch.Size([4, 4])
print("Raw attention scores (rows = querying token, cols = token being looked at):")
print(scores.round(decimals=2))

# Reading it: scores[2, 1] is how strongly "bank" (row 2) matches "river" (col 1).
# These raw numbers can be any size, positive or negative -- we fix that next.

**What this does:** It computes a `(4, 4)` matrix of raw match scores. Entry `scores[i, j]` is the dot product of token *i*'s query with token *j*'s key — a single number saying "how relevant is *j* to *i*?". Row 2 (`"bank"`) tells us how much `"bank"` matches each other token. The numbers are unbounded and hard to interpret directly, so the next two steps clean them up.

### ✏️ Exercise
Print just `scores[2]` — the row for `"bank"`. Which column has the largest value? With random weights it could be anything, but the *machinery* is what matters: after training, a real model would learn weights that make `"bank"`'s row light up on `"river"`.

## 5. Step 2 — scale by √d to keep numbers calm

There's a subtle but important detail. When the vectors are long (high-dimensional), dot products tend to become **large** numbers. Large scores make the next step (softmax) produce extreme, "all-or-nothing" weights, which makes training unstable.

The fix is simple: **divide every score by the square root of the dimension** (`√d`). This is the "**scaled**" in *scaled dot-product attention*. It keeps the scores in a comfortable range no matter how big `d` is. Here `d = 8`, so we divide by `√8 ≈ 2.83`.

In [ ]:
d_k = K.shape[1]                 # the dimension of each key vector (8 here)
scaled_scores = scores / math.sqrt(d_k)   # divide every score by sqrt(8) ~= 2.83

print("sqrt(d_k):", round(math.sqrt(d_k), 3))   # -> sqrt(d_k): 2.828
print("Scaled scores (smaller, calmer numbers):")
print(scaled_scores.round(decimals=2))

**What this does:** It divides the whole score matrix by `√8`. Notice the numbers shrink toward zero compared to the previous cell. This single division is the entire "scaling" trick — cheap to do, and it keeps softmax well-behaved so the model can learn smoothly.

### ✏️ Exercise
What would `d_k` be if our embedding dimension were 64 (a common small size)? Compute `math.sqrt(64)`. Bigger `d` means we divide by a bigger number — that's the whole point: keep scores tame as models grow.

## 6. Step 3 — softmax: turn scores into weights that sum to 1

We now have calmed-down scores, but we want **attention weights**: a set of numbers per token that are all **positive** and **add up to 1**, so we can use them to take a weighted average. That's exactly what **softmax** does.

In plain words, softmax:

1. Makes every number positive (using the exponential function — bigger scores become *much* bigger).
2. Divides each by the total, so the whole row sums to **1**.

The result reads like percentages: "spend 70% of your attention on this token, 20% on that one, 10% on the rest." We apply softmax **across each row** (`dim=-1`), because each row is one token deciding how to split its attention over all tokens.

In [ ]:
# A 3-number warm-up so you can SEE what softmax does.
demo = torch.tensor([2.0, 1.0, 0.1])
print("input scores :", demo)
print("after softmax:", F.softmax(demo, dim=-1).round(decimals=3))
print("they sum to  :", F.softmax(demo, dim=-1).sum().item())   # -> 1.0
print()

# Now the real thing: softmax across each ROW of the scaled score matrix.
# dim=-1 means "normalize along the last axis" = across columns within each row.
attn_weights = F.softmax(scaled_scores, dim=-1)

print("Attention weights (each row sums to 1):")
print(attn_weights.round(decimals=2))
print("Row sums:", attn_weights.sum(dim=-1).round(decimals=2))   # -> all 1.00

**What this does:** The warm-up shows softmax turning `[2.0, 1.0, 0.1]` into something like `[0.66, 0.24, 0.10]` — positive numbers that sum to 1, with the biggest input getting the biggest share. Then we apply it to every row of our scaled scores to get `attn_weights`, a `(4, 4)` matrix where **each row is one token's attention budget**, split across all 4 tokens and summing to 1.

### ✏️ Exercise
Change the warm-up `demo` to `[5.0, 1.0, 0.1]` and re-run. Notice how the first weight gets *much* closer to 1 — softmax exaggerates differences. That's why scaling (the previous step) matters: it stops scores from getting so large that one token grabs *all* the attention.

## 7. Step 4 — mix the Values using those weights

Last step. Each token now has an **attention weight** for every token (how much to listen to it). To build the token's **new, context-aware representation**, we take a **weighted average of all the Value vectors**, using those weights.

Concretely: `output = attn_weights @ V`. Because `attn_weights` is `(4, 4)` and `V` is `(4, 8)`, the output is `(4, 8)` — same shape as we started with, but now **every token's vector has been blended with information from the tokens it attended to.** `"bank"`'s new vector literally contains a dose of `"river"`'s value if it attended to `"river"`.

In [ ]:
# Weighted sum of value vectors: (4, 4) @ (4, 8) -> (4, 8).
output = attn_weights @ V

print("output shape:", output.shape)   # -> output shape: torch.Size([4, 8])
print("New context-aware vector for 'bank' (row 2):")
print(output[2].round(decimals=2))

# Sanity check: this is the SAME shape as the input X (4, 8). Attention takes
# token vectors in and returns context-mixed token vectors out -- ready to feed
# into the next part of the Transformer block.
print("input X shape :", X.shape, "  output shape:", output.shape)

**What this does:** It produces `output`, the result of one full attention operation. `output[2]` is `"bank"` *after* it has absorbed information from the tokens it paid attention to. Crucially, `output` has the **same shape** as the input `X` (`(4, 8)`), which is what lets us stack many attention layers in a row — each one's output is the next one's input.

### ✏️ Exercise
Compare `X[2]` (original `"bank"`) with `output[2]` (context-aware `"bank"`). They're different vectors — attention changed the representation. In one sentence, explain *why* we'd want `"bank"`'s vector to change based on its neighbors.

### Visualizing attention: who looks at whom?

The attention weight matrix is the most interpretable part of a Transformer. Let's draw it as a heatmap. Each **row** is a token doing the looking; each **column** is a token being looked at; **brighter = more attention**.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
heat = ax.imshow(attn_weights.detach().numpy(), cmap="viridis")

# Label the axes with our actual words so the picture is readable.
ax.set_xticks(range(n_tokens)); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(n_tokens)); ax.set_yticklabels(tokens)
ax.set_xlabel("token being looked at (Key)")
ax.set_ylabel("querying token (Query)")
ax.set_title("Attention weights")

# Write the actual weight in each cell so you can read exact numbers too.
for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f"{attn_weights[i, j]:.2f}", ha="center", va="center", color="white")

fig.colorbar(heat, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

**What this does:** It renders the `(4, 4)` attention matrix as a labeled heatmap. Read it **row by row**: row `"bank"` shows how `"bank"` split its attention over `the / river / bank / flooded`. With our *random* weights the pattern is meaningless — but this exact picture, drawn for a *trained* model, is how researchers see things like "this attention head links pronouns to the nouns they refer to."

### ✏️ Exercise
Look at the `"bank"` row in the heatmap. Which token does it attend to most? Re-run the notebook from the top with a different `torch.manual_seed` (say `7`) and watch the pattern change — a reminder that *meaningful* attention patterns come from **training**, not random initialization.

## 8. Putting it together: scaled dot-product attention in one function

We just did four steps: **scores → scale → softmax → weighted values**. Let's bundle them into a single reusable function. This is essentially the exact operation inside every Transformer (PyTorch even ships a built-in version, `F.scaled_dot_product_attention`).

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """One attention operation. Q, K, V are each (n_tokens, d). Returns (output, weights)."""
    d_k = K.shape[-1]                       # size of each key vector
    scores = Q @ K.transpose(-2, -1)        # (n, d) @ (d, n) -> (n, n): all query-key matches
    scores = scores / math.sqrt(d_k)        # scale to keep numbers calm
    weights = F.softmax(scores, dim=-1)     # rows -> attention weights summing to 1
    output  = weights @ V                   # blend the values: (n, n) @ (n, d) -> (n, d)
    return output, weights

# Verify it matches what we computed by hand earlier.
out2, w2 = scaled_dot_product_attention(Q, K, V)
print("Matches our step-by-step output?", torch.allclose(out2, output))   # -> True
print("Matches our step-by-step weights?", torch.allclose(w2, attn_weights))  # -> True

**What this does:** It packages the four steps into `scaled_dot_product_attention(Q, K, V)` and confirms (with `torch.allclose`, which checks two tensors are equal up to tiny rounding) that it reproduces our hand-computed result exactly. `K.transpose(-2, -1)` is the general way to write "swap the last two axes" — the same as `K.T` for a 2-D tensor, but it also works when there's a batch dimension.

### ✏️ Exercise
Call the function on a *new* random input: make `X2 = torch.randn(3, 8)`, derive `Q2, K2, V2` with the existing weight matrices, and print the returned `weights`. Confirm it's `(3, 3)` and each row sums to 1.

## 9. Multi-head attention: several attentions at once

One attention operation can only learn **one kind** of relationship at a time. But language has many simultaneous relationships — grammar, subject-verb links, what a pronoun refers to, tone, and more.

**Multi-head attention** solves this by running several attention operations **in parallel**, called **heads**. Each head gets its own `W_q`, `W_k`, `W_v`, so each can specialize: one head might learn to link adjectives to nouns, another might track long-range references. The model splits the embedding into chunks (one per head), runs attention on each chunk, then **concatenates** the results back together. That's it — same math as before, just several copies side by side.

In [ ]:
# A tiny sketch: 2 heads, each working on a 4-dim slice of our 8-dim vectors.
n_heads   = 2
d_head    = d_model // n_heads   # 8 / 2 = 4 dims per head

head_outputs = []
for h in range(n_heads):
    # Each head gets its OWN random projection matrices (8 -> 4).
    Wq_h = torch.randn(d_model, d_head)
    Wk_h = torch.randn(d_model, d_head)
    Wv_h = torch.randn(d_model, d_head)
    Qh, Kh, Vh = X @ Wq_h, X @ Wk_h, X @ Wv_h     # each (4, 4)
    out_h, _ = scaled_dot_product_attention(Qh, Kh, Vh)   # attention within this head
    head_outputs.append(out_h)
    print(f"head {h} output shape:", out_h.shape)   # -> (4, 4)

# Concatenate the heads back along the feature axis: two (4, 4) -> one (4, 8).
multi_head_output = torch.cat(head_outputs, dim=-1)
print("multi-head output shape:", multi_head_output.shape)   # -> (4, 8)

**What this does:** It runs **2 heads** in a loop. Each head projects the 8-dim tokens down to a 4-dim slice, does its own attention, and produces a `(4, 4)` output. We `torch.cat` the two head outputs back into a `(4, 8)` tensor — the same shape we started with. Real models use the same pattern but run the heads in one batched operation for speed, and add a final mixing matrix. The takeaway: **more heads = more relationships the model can track at once.**

### ✏️ Exercise
Set `n_heads = 4` and re-run. What is `d_head` now? (Hint: `8 // 4`.) Confirm the final `multi_head_output` is still `(4, 8)` — the number of heads changes how the work is *divided*, not the final shape.

## 10. The Transformer block: the unit we stack

Attention is the star, but a full **Transformer block** wraps it with a few more pieces. Here's the standard recipe, as a diagram:

```
          input tokens (4 x 8)
                 │
        ┌────────┴─────────┐
        │  Multi-Head      │
        │  Attention       │   <- tokens look at each other
        └────────┬─────────┘
                 │
            (+) add input back   <- RESIDUAL connection (a shortcut)
                 │
            Layer Norm           <- rescales numbers to keep training stable
                 │
        ┌────────┴─────────┐
        │  Feed-Forward    │   <- a small 2-layer network applied to EACH token
        │  Network (MLP)   │      (think on each token individually)
        └────────┬─────────┘
                 │
            (+) add input back   <- RESIDUAL connection again
                 │
            Layer Norm
                 │
          output tokens (4 x 8)   <- same shape in, same shape out -> STACKABLE
```

Three supporting ideas in plain English:

- **Feed-forward network (FFN/MLP):** after tokens have shared information via attention, each token is passed *individually* through a small 2-layer neural network so the model can "think" about what it gathered. This is where a large share of a model's parameters live.
- **Residual connection (the "+ add input back"):** we add the block's input to its output. This gives gradients a clean shortcut and lets us stack *many* blocks without the signal getting lost. ("Don't replace what you had — *add* to it.")
- **Layer norm:** rescales each token's vector to a stable range so numbers don't blow up or vanish as they flow through dozens of layers.

Because a block takes `(4, 8)` in and gives `(4, 8)` out, we can **stack** blocks: 12 of them for a small model, 80+ for a large one. **A "large language model" is essentially this block repeated many times.**

In [ ]:
import torch.nn as nn

class TinyTransformerBlock(nn.Module):
    """A minimal Transformer block: attention -> add+norm -> FFN -> add+norm."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        # PyTorch's built-in multi-head attention (batch_first => shape (batch, tokens, d)).
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)          # layer norm after attention
        self.norm2 = nn.LayerNorm(d_model)          # layer norm after the FFN
        self.ffn = nn.Sequential(                   # the small 2-layer feed-forward net
            nn.Linear(d_model, 4 * d_model),        # expand (common to use 4x width)
            nn.ReLU(),                              # non-linearity
            nn.Linear(4 * d_model, d_model),        # project back to d_model
        )

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)            # self-attention: query=key=value=x
        x = self.norm1(x + attn_out)                # residual add, then normalize
        x = self.norm2(x + self.ffn(x))             # residual add around the FFN, normalize
        return x

block = TinyTransformerBlock(d_model=8, n_heads=2)
x_batch = X.unsqueeze(0)                            # add a batch dim: (4,8) -> (1,4,8)
out_block = block(x_batch)
print("block input shape :", x_batch.shape)        # -> torch.Size([1, 4, 8])
print("block output shape:", out_block.shape)      # -> torch.Size([1, 4, 8]) (same!)

# Count the learnable weights -- THIS is what fine-tuning adjusts.
n_params = sum(p.numel() for p in block.parameters())
print("trainable parameters in ONE tiny block:", n_params)

**What this does:** It builds a real (if tiny) Transformer block using PyTorch's own layers and runs our toy sentence through it. Notice the output shape **equals** the input shape `(1, 4, 8)` — that's the stackability we keep stressing. The final line counts the block's **trainable parameters**: every one of those numbers is a knob that **fine-tuning can turn**. A real LLM has billions of such knobs spread across dozens of these blocks.

### ✏️ Exercise
Stack two blocks: create `block2 = TinyTransformerBlock(8, 2)` and run `out_block` through it (`block2(out_block)`). Confirm the shape is still `(1, 4, 8)`. You just built a 2-layer Transformer — real models are the same idea, scaled up.

## 11. Encoder vs decoder, and causal masking

The same block comes in two famous flavors, depending on **which tokens are allowed to attend to which**:

- **Encoder (BERT-style):** every token may attend to **every** token — both left and right. The model sees the *whole* sentence at once, which is great for **understanding** tasks: classification, sentiment, search, filling in a blanked-out word. Encoders read; they don't generate text left-to-right.
- **Decoder (GPT-style):** each token may attend only to itself and the tokens **before** it — never the future. This is enforced by a **causal mask** that hides upcoming tokens. It's exactly what you need for **generation**: to predict the next word, the model must not be allowed to peek at it. GPT, Llama, Mistral, and most chat models are decoders.

**Causal masking** in one breath: before softmax, we set the scores for "future" positions to negative infinity, so softmax assigns them ~0 weight. The token literally cannot see ahead. Let's make that mask visible.

In [ ]:
# A causal mask for 4 tokens: True where a token is ALLOWED to look.
# Lower-triangular = "you may see yourself and everything before you."
mask = torch.tril(torch.ones(n_tokens, n_tokens)).bool()
print("Causal mask (True = allowed to attend):")
print(mask)

# Apply it to our earlier scaled scores: forbidden spots become -inf, so after
# softmax they get ~0 weight. Token 0 sees only itself; token 3 sees all four.
masked_scores = scaled_scores.masked_fill(~mask, float("-inf"))
causal_weights = F.softmax(masked_scores, dim=-1)

print()
print("Causal attention weights (note the zeros in the upper-right triangle):")
print(causal_weights.round(decimals=2))

**What this does:** It builds a lower-triangular mask and applies it. In the printed `causal_weights`, the **upper-right triangle is all zeros**: token 0 (`"the"`) can only attend to itself; token 3 (`"flooded"`) can attend to all four. This is the single difference that turns a "read everything" encoder into a "predict the next word" decoder — and it's why GPT-style models can generate text one token at a time.

### ✏️ Exercise
Look at row 0 of `causal_weights`. Why is it `[1, 0, 0, 0]`? (Hint: the first token has only one token it's allowed to attend to — itself — and softmax of a single allowed value is 1.) Then explain in one sentence why an *encoder* would **not** want this mask.

## 12. Tying it back to fine-tuning

Let's connect every piece to what you'll actually do later in this course:

- **An LLM is a tall stack of the block from Section 10.** Embeddings go in the bottom, flow up through dozens of attention + feed-forward blocks, and predictions come out the top.
- **Fine-tuning changes the weights *inside* those blocks** — mainly the attention projection matrices (`W_q`, `W_k`, `W_v`, and the output mixing matrix) and the feed-forward `Linear` layers. Those are precisely the `parameters` we counted in Section 10.
- **Full fine-tuning** updates *all* of them: maximum flexibility, maximum memory cost.
- **LoRA (coming later)** freezes the originals and inserts **small extra trainable matrices** next to the attention and feed-forward weights. Far fewer numbers to train, far less memory — and now you know *exactly where those extra pieces attach*, because you've seen the layers they attach to.
- **Encoder vs decoder tells you which model to pick:** fine-tuning for **classification**? Reach for an encoder (BERT-family). Fine-tuning for **chat or text generation**? Reach for a decoder (GPT/Llama-family).

You don't need to memorize the math. Carry the picture: **tokens → vectors → attention lets them share context → feed-forward thinks on each one → stack it deep → fine-tuning nudges the weights inside.**

In [ ]:
# A back-of-envelope feel for scale. We won't build this -- just do the arithmetic.
small_llm_layers = 32      # a small-ish modern LLM might have ~32 blocks
params_per_block = n_params # from our TINY d_model=8 block in Section 10

print("Our toy block params      :", params_per_block)
print("Toy 'stack' of 32 blocks  :", params_per_block * small_llm_layers)
print()
print("Real models scale d_model into the thousands and stack dozens of blocks,")
print("which is how parameter counts reach the billions. Fine-tuning adjusts")
print("those weights (full FT) or adds small ones beside them (LoRA).")

**What this does:** It does a tiny multiplication to give you a *feel* for how parameter counts explode: even our toy block, repeated 32 times, already has a meaningful count — and real models use far larger `d_model` and more layers. The point isn't the exact number; it's the mental model of **stacking**, which is what makes fine-tuning both powerful and (for full fine-tuning) expensive.

### ✏️ Exercise
In one or two sentences of your own words, explain to an imaginary friend: "When I fine-tune an LLM, what is actually changing inside it?" If you can answer that clearly, this notebook did its job.

## Common mistakes & how to debug them

- **Shape mismatch in `Q @ K.T`.** The number one error. If you get a size error, **print the shapes** of `Q` and `K` first. The rule: `(n, d) @ (d, n)` works; you need `K.T` (or `K.transpose(-2, -1)`) so the inner dimensions match.
- **Softmax over the wrong axis.** Use `dim=-1` so *each token's row* sums to 1. If you softmax over `dim=0` (columns), the math runs but the meaning is wrong — always check that **rows** sum to 1 with `weights.sum(dim=-1)`.
- **Forgetting to scale by √d.** Without it, scores can get large, softmax saturates (one weight ≈ 1, the rest ≈ 0), and the model attends to only one token. Symptom: attention heatmaps that are a single bright dot per row.
- **Confusing encoder and decoder.** Trying to *generate* text with an encoder, or applying a causal mask to a classification model, gives bad results. Match the architecture to the task.
- **Causal mask using the wrong triangle.** A decoder needs a **lower**-triangular mask (`torch.tril`). If you accidentally mask the lower triangle, every token sees only the *future* — backwards and broken.
- **Mixing up `.T` on batched tensors.** `K.T` only does the right thing for a plain 2-D tensor. With a batch dimension, use `K.transpose(-2, -1)` so you swap only the last two axes.
- **Reading the heatmap backwards.** Rows = the token doing the looking (query); columns = the token being looked at (key). Label your axes (as we did) so you never get it backwards.

## Summary

- **Attention solves context:** it lets each token look at every other token, so meaning that depends on neighbors (like "bank" near "river") is captured.
- **Query / Key / Value** is a *soft dictionary lookup*: queries ask, keys advertise, values carry the content; match scores decide the mix.
- **Scaled dot-product attention** is four steps: `scores = Q @ K.T` → divide by `√d` → `softmax` (rows sum to 1) → multiply by `V`. Same shape out as in.
- **Softmax** turns raw scores into positive weights that sum to 1 — like attention "percentages".
- **Multi-head attention** runs several attentions in parallel so the model tracks several relationships at once.
- **A Transformer block** = multi-head attention + feed-forward network, each wrapped in a **residual connection** and **layer norm**. Same shape in and out, so blocks **stack**.
- **Encoder (BERT)** sees all tokens → understanding/classification. **Decoder (GPT)** uses a **causal mask** to hide the future → generation.
- **Fine-tuning connection:** an LLM is a tall stack of these blocks; fine-tuning adjusts the weights inside the attention and feed-forward layers (and **LoRA** later adds small trainable pieces beside them).

## What to learn next

Next up: **`07_tokenization_and_embeddings.ipynb`**. We waved our hands and *invented* random embeddings (`X = torch.randn(...)`) in this notebook. The next one fills that gap: how raw text becomes **tokens**, and how tokens become the **embedding vectors** that feed straight into the attention we just built. You'll see where the very first numbers in the model come from — and why tokenization quietly shapes everything fine-tuning does afterward.

See you in notebook 07!